In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm


In [13]:
import os, cv2, random, numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

def corrupt_image(img):
    def add_noise(i):
        return np.clip(i+np.random.normal(0,20,i.shape),0,255).astype(np.uint8)
    def blur(i):
        return cv2.GaussianBlur(i,(5,5),0)
    def lowres(i):
        h,w=i.shape[:2]
        return cv2.resize(cv2.resize(i,(w//3,h//3)),(w,h))
    return random.choice([add_noise,blur,lowres])(img)

class RAImageDataset(Dataset):
    def __init__(self, root_dir, img_size=224):
        self.paths=[os.path.join(root_dir,f) for f in os.listdir(root_dir)]
        self.transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((img_size,img_size))
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self,idx):
        img=cv2.imread(self.paths[idx])
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
        corrupted=corrupt_image(img.copy())
        return self.transform(corrupted),self.transform(img),0


In [14]:
DATA_PATH = r"D:\Image Recognition\data"
dataset = RAImageDataset(DATA_PATH)
loader = DataLoader(dataset,batch_size=8,shuffle=True)


In [15]:
class Block(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(a,b,3,1,1),
            nn.BatchNorm2d(b),
            nn.ReLU())
    def forward(self,x): return self.net(x)

class ProcessingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            Block(3,64),Block(64,64),
            nn.MaxPool2d(2),
            Block(64,128),Block(128,128),
            nn.Upsample(scale_factor=2),
            Block(128,64),
            nn.Conv2d(64,3,3,1,1),
            nn.Sigmoid())
    def forward(self,x): return self.net(x)


In [16]:
device="cuda" if torch.cuda.is_available() else "cpu"

model=ProcessingNet().to(device)
loss_fn=nn.MSELoss()
opt=optim.Adam(model.parameters(),lr=1e-3)

for epoch in range(10):
    total=0
    pbar=tqdm(loader)
    for x,y,_ in pbar:
        x,y=x.to(device),y.to(device)

        out=model(x)
        loss=loss_fn(out,y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total+=loss.item()
        pbar.set_postfix(loss=loss.item())

    print("Epoch loss:",total/len(loader))


100%|██████████| 375/375 [02:26<00:00,  2.57it/s, loss=0.00307]


Epoch loss: 0.006923617320756118


100%|██████████| 375/375 [02:13<00:00,  2.80it/s, loss=0.00236]


Epoch loss: 0.004893057309401532


100%|██████████| 375/375 [02:12<00:00,  2.84it/s, loss=0.00258]


...

Epoch loss: 0.0025261340389649074


100%|██████████| 375/375 [02:19<00:00,  2.69it/s, loss=0.00209] 


Epoch loss: 0.002400073745287955


100%|██████████| 375/375 [02:15<00:00,  2.76it/s, loss=0.0018]  

Epoch loss: 0.0023536894880235196


In [17]:
torch.save(model.state_dict(),"baseline_model.pth")
print("Saved model")
    

Saved model
